# 01. Exploratory Data Analysis & MIND Dataset Preparation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TheNarhawl/mind-dataset-recsys/blob/main/notebooks/01_eda_and_data_prep.ipynb)

This notebook covers:
1. Downloading and extracting the **MIND-small** dataset (train & dev).
2. Exploratory Data Analysis: News categories, user reading history lengths, impression CTR.
3. Temporal validation splitting (preventing future information leakage).
4. Generating news text representations and preprocessed dataframes for Stage 1 & Stage 2.


In [ ]:
# Install mind-recsys package
!pip install -q uv
!git clone https://github.com/TheNarhawl/mind-dataset-recsys.git
%cd mind-dataset-recsys
!pip install -q -e .


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from mind_recsys.data.download import download_mind_small
from mind_recsys.data.dataset import load_news, load_behaviors, MINDDataset

data_dir = Path("./data")
download_mind_small(data_dir, include_dev=True)

dataset = MINDDataset(data_dir)
train_news = dataset.load_train_news()
train_behaviors = dataset.load_train_behaviors()
dev_news = dataset.load_dev_news()
dev_behaviors = dataset.load_dev_behaviors()

print(f"Train news: {len(train_news):,}, Train impressions: {len(train_behaviors):,}")
print(f"Dev news: {len(dev_news):,}, Dev impressions: {len(dev_behaviors):,}")


In [ ]:
# Category Distribution
plt.figure(figsize=(12, 5))
train_news['category'].value_counts().head(15).plot(kind='bar', color='#3498db')
plt.title('Top-15 News Categories in MIND-small')
plt.xlabel('Category')
plt.ylabel('Count')
plt.grid(axis='y', alpha=0.3)
plt.show()


In [ ]:
# User History Length Distribution
train_behaviors['hist_len'] = train_behaviors['parsed_history'].apply(len)
print(train_behaviors['hist_len'].describe())

plt.figure(figsize=(10, 4))
sns.histplot(train_behaviors['hist_len'], bins=50, kde=True, color='#2ecc71')
plt.title('Distribution of User Reading History Length')
plt.xlabel('Number of Clicked News in History')
plt.xlim(0, 150)
plt.show()


In [ ]:
# Impressions and CTR Analysis
all_clicks = sum(train_behaviors['parsed_impressions'].apply(lambda imps: sum(c[1] for c in imps)))
all_imps = sum(train_behaviors['parsed_impressions'].apply(len))
print(f"Total shown candidates: {all_imps:,}")
print(f"Total positive clicks: {all_clicks:,}")
print(f"Overall CTR: {all_clicks / all_imps:.4%}")


In [ ]:
# Export clean preprocessed data
processed_dir = data_dir / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
train_news.to_parquet(processed_dir / "train_news.parquet")
train_behaviors.to_parquet(processed_dir / "train_behaviors.parquet")
dev_news.to_parquet(processed_dir / "dev_news.parquet")
dev_behaviors.to_parquet(processed_dir / "dev_behaviors.parquet")
print("Preprocessed parquet files saved successfully!")
